# Week 3 - Preprocessing and Evaluation Pipeline

## Purpose

This notebook prepares the common data-preprocessing and evaluation pipeline for the multi-label emotion classification experiments.

The main tasks are:

1. Convert emotion labels into multi-hot vectors.
2. Implement reusable text preprocessing functions.
3. Prepare train, validation, and test data in a consistent format.
4. Implement common evaluation metrics for later model comparison.

In [1]:
import sys
from pathlib import Path
import re

import numpy as np
import pandas as pd

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score,
    hamming_loss,
    average_precision_score,
)

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"

FIGURE_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"

SRC_DIR = PROJECT_ROOT / "src"


sys.path.append(str(SRC_DIR))

from utils.labels import LABELS

train_path = DATA_DIR / "train.tsv"
val_path = DATA_DIR / "val.tsv"
test_path = DATA_DIR / "test.tsv"

train_df = pd.read_csv(train_path, sep="\t", header=None, names=["id", "text", "labels"])
val_df = pd.read_csv(val_path, sep="\t", header=None, names=["id", "text", "labels"])
test_df = pd.read_csv(test_path, sep="\t", header=None, names=["id", "text", "labels"])

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

train_df.head()

(40000, 3)
(5000, 3)
(5000, 3)


,id,text,labels
0,39087,내가 톰행크스를 좋아하긴 했나보다... 초기 영화 빼고는 다 봤네.,"2,13,15,16,29,39"
1,30893,"정말 상상을 초월하는 무개념 진상들 상대하다 우울증, 공항장애 걸리는 공무원 많아요...","0,5,7,10,19,22,29,35,36,38"
2,45278,"새로운 세상과 조우한 자의 어린아이 같은 반응, 어쩌면 회복된 것은 눈이 아닌 순수...","1,2,7"
3,16398,미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ,"9,15,20,23,26,28,29"
4,13653,네 맞습니다 플스는 역시 30프레임이 어울리죠 ㅎ,"1,2,8,9,11,13,15,16,28,29,32,40,42"


In [2]:
def parse_label_ids(label_string):
    return [
        int(label_id.strip())
        for label_id in label_string.split(",")
    ]

In [3]:
for df in [train_df, val_df, test_df]:
    df["label_ids"] = df["labels"].apply(parse_label_ids)

train_df[["labels", "label_ids"]].head()

,labels,label_ids
0,"2,13,15,16,29,39","[2, 13, 15, 16, 29, 39]"
1,"0,5,7,10,19,22,29,35,36,38","[0, 5, 7, 10, 19, 22, 29, 35, 36, 38]"
2,"1,2,7","[1, 2, 7]"
3,"9,15,20,23,26,28,29","[9, 15, 20, 23, 26, 28, 29]"
4,"1,2,8,9,11,13,15,16,28,29,32,40,42","[1, 2, 8, 9, 11, 13, 15, 16, 28, 29, 32, 40, 42]"


In [4]:
mlb = MultiLabelBinarizer(
    classes=list(range(len(LABELS)))
)

y_train = mlb.fit_transform(train_df["label_ids"])
y_val = mlb.transform(val_df["label_ids"])
y_test = mlb.transform(test_df["label_ids"])

print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

print(train_df["label_ids"].iloc[0])
print(y_train[0])

(40000, 44)
(5000, 44)
(5000, 44)
[2, 13, 15, 16, 29, 39]
[0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0
 0 0 1 0 0 0 0]


In [5]:
X_train = train_df["text"].copy()
X_val = val_df["text"].copy()
X_test = test_df["text"].copy()

In [6]:
def preprocess_original(text):
    return text

def normalize_repeated_characters(text, max_repeat=2):
    pattern = r"(.)\1{" + str(max_repeat) + r",}"
    return re.sub(
        pattern,
        lambda match: match.group(1) * max_repeat,
        text,
    )

def remove_selected_symbols(text):
    text = re.sub(
        r"[^\w\s가-힣!?~ㅋㅎㅠㅜ]",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


In [7]:
sample = "ㅋㅋㅋㅋㅋㅋ 너무 좋다아아아아 :) ^^!"
print(normalize_repeated_characters(sample))
print(remove_selected_symbols(sample))

ㅋㅋ 너무 좋다아아 :) ^^!
ㅋㅋㅋㅋㅋㅋ 너무 좋다아아아아 !


In [8]:
def preprocess_text(
        text,
        normalize_repeat=False,
        remove_symbols=False,
):
    processed = text

    if normalize_repeat:
        processed = normalize_repeated_characters(processed)
    
    if remove_symbols:
        processed = remove_selected_symbols(processed)
    
    return processed


In [9]:
sample = "ㅋㅋㅋㅋ좋아아아아아!!!!!!!^^> 오오오오:)"

print(preprocess_text(sample, normalize_repeat=True, remove_symbols=True))

ㅋㅋ좋아아!! 오오


In [10]:
sample_texts = train_df["text"].head(10)

for text in sample_texts:
    print("Original:")
    print(text)

    print("Normalized:")
    print(preprocess_text(text, normalize_repeat=True))

    print("Normalized + Symbol Removal:")
    print(preprocess_text(text, normalize_repeat=True, remove_symbols=True))

    print("-" * 50)


Original:
내가 톰행크스를 좋아하긴 했나보다... 초기 영화 빼고는 다 봤네.
Normalized:
내가 톰행크스를 좋아하긴 했나보다.. 초기 영화 빼고는 다 봤네.
Normalized + Symbol Removal:
내가 톰행크스를 좋아하긴 했나보다 초기 영화 빼고는 다 봤네
--------------------------------------------------
Original:
정말 상상을 초월하는 무개념 진상들 상대하다 우울증, 공항장애 걸리는 공무원 많아요 ㅠ 생각보다 스트레스 심한 직업군이라는 생각입니다.
Normalized:
정말 상상을 초월하는 무개념 진상들 상대하다 우울증, 공항장애 걸리는 공무원 많아요 ㅠ 생각보다 스트레스 심한 직업군이라는 생각입니다.
Normalized + Symbol Removal:
정말 상상을 초월하는 무개념 진상들 상대하다 우울증 공항장애 걸리는 공무원 많아요 ㅠ 생각보다 스트레스 심한 직업군이라는 생각입니다
--------------------------------------------------
Original:
새로운 세상과 조우한 자의 어린아이 같은 반응, 어쩌면 회복된 것은 눈이 아닌 순수함일지도 모르겠습니다. 그걸 가능케 한 사랑에 경의를 표합니다.
Normalized:
새로운 세상과 조우한 자의 어린아이 같은 반응, 어쩌면 회복된 것은 눈이 아닌 순수함일지도 모르겠습니다. 그걸 가능케 한 사랑에 경의를 표합니다.
Normalized + Symbol Removal:
새로운 세상과 조우한 자의 어린아이 같은 반응 어쩌면 회복된 것은 눈이 아닌 순수함일지도 모르겠습니다 그걸 가능케 한 사랑에 경의를 표합니다
--------------------------------------------------
Original:
미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ
Normalized:
미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋ
Normalized +

In [11]:
def evaluate_multilabel(y_true, y_pred, y_prob=None):
    results = {}

    results["micro_f1"] = f1_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0,
    )

    results["macro_f1"] = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    results["hamming_losss"] = hamming_loss(
        y_true,
        y_pred,
    )

    if y_prob is not None:
        results["pr_auc_macro"] = average_precision_score(
            y_true,
            y_prob,
            average="macro",
        )

        results["pr_auc_micro"] = average_precision_score(
            y_true,
            y_prob,
            average="micro",
        )
    
    return results

In [12]:
def get_label_wise_f1(y_true, y_pred):
    scores = f1_score(
        y_true,
        y_pred,
        average=None,
        zero_division=0
    )

    return pd.DataFrame({
        "label_id": range(len(LABELS)),
        "label": LABELS,
        "f1": scores,
    })

In [13]:
def apply_threshold(y_prob, threshold=0.5):
    return (y_prob >= threshold).astype(int)

In [14]:
y_true_dummy = np.array([
    [1, 0, 1],
    [0, 1, 0],
])

y_prob_dummy = np.array([
    [0.8, 0.2, 0.7],
    [0.3, 0.9, 0.1],
])

y_pred_dummy = apply_threshold(
    y_prob_dummy,
    threshold=0.5,
)

print(y_pred_dummy)

[[1 0 1]
 [0 1 0]]


In [15]:
print(evaluate_multilabel(y_true_dummy, y_pred_dummy, y_prob_dummy))

{'micro_f1': 1.0, 'macro_f1': 1.0, 'hamming_losss': 0.0, 'pr_auc_macro': 1.0, 'pr_auc_micro': 1.0}


In [16]:
print("X_train:", len(X_train))
print("X_val:", len(X_val))
print("X_test:", len(X_test))

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: 40000
X_val: 5000
X_test: 5000
y_train: (40000, 44)
y_val: (5000, 44)
y_test: (5000, 44)


## Reusable Functions

The functions implemented above were saved in `src/` and will be reused from this point.

In [17]:
import preprocessing.labels as label_prep
import preprocessing.text as text_prep
import evaluation.metrics as eval_metrics

print(label_prep.parse_label_ids("0, 9, 22"))

sample_text = "ㅋㅋㅋㅋㅋㅋ좋아아아아ㅏㅏㅏㅏㅏㅏ@^@:)"

print(text_prep.preprocess_text(sample_text, normalize_repeat=True, remove_symbols=True))
print()

y_pred_dummy = eval_metrics.apply_threshold(y_prob_dummy, threshold=0.5)

print(eval_metrics.evaluate_multilabel(y_true_dummy, y_pred_dummy, y_prob_dummy))

[0, 9, 22]
ㅋㅋ좋아아ㅏㅏ

{'micro_f1': 1.0, 'macro_f1': 1.0, 'hamming_losss': 0.0, 'pr_auc_macro': 1.0, 'pr_auc_micro': 1.0}
